# Validation of circumpolar boreal vegetation height and AGBD, v3.1 (2016–2025)

One notebook that tells the field-plot validation story for the v3.1 multi-year **vegetation height** (`Ht_H30_YYYY_v3.1_multiyr`) and **aboveground biomass density** (`AGB_H30_YYYY_v3.1_multiyr`) time series.

**The core problem.** Most reference plots were measured years to decades before 2016–2025 (Canadian NFI 1992–2007, Eurasia/NASA 2002–2016, Miesner 2011–2021, Kolyma 2010–2019). A field plot is only a valid reference for a map year if the stand has not been structurally altered in between (fire, harvest, other stand-replacing events) and has not grown appreciably. We use the 2020 **forest stand age** map (`value_CACC_2020_v3.1_multiyr_age_mean`, `_age_std`) to decide which plot × map-year pairs are still usable.

**Story, in order**
1. Reference data: what was measured, where, when, and how each source defines height and AGBD.
2. Stand-age screening: which plots were disturbed after measurement, which are too young to be stable across the gap, and what the screen removes.
3. Height validation (all → age-screened → strict), by source, region, and plot group.
4. AGBD validation, same structure.
5. Time-series consistency: accuracy per map year on a fixed plot set; bias vs. time gap; per-pixel uncertainty coverage.
6. Sensitivity of the conclusions to every screening threshold.

**Inputs** are the `*_20260513` extractions made by `extract_rasters_to_val_plots.ipynb` (v3.1 height and AGBD 2016–2025 mean/std, CACC 2020 age, AGB 2020–2025 trend). Eurasia/NASA plot heights are joined from the `smrytrees` extraction by `site`.

**Companion files**: `VALIDATION_v3.1_LOG.md` (method log, decisions, trade-offs) and `validation_v3.1_lidar_LVIS_GLiHT.ipynb` (airborne lidar validation).

--------------------------------------------------------------------------------
Paul Montesano, PhD | NASA GSFC, ESSIC
--------------------------------------------------------------------------------

In [1]:
suppressPackageStartupMessages({
    library(sf)
    library(dplyr)
    library(tidyr)
    library(stringr)
    library(purrr)
    library(readr)
    library(ggplot2)
    library(patchwork)
})
sf_use_s2(FALSE)
options(dplyr.summarise.inform = FALSE, width = 200)

Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘tidyr’ was built under R version 4.4.3”
Warning message:
“package ‘stringr’ was built under R version 4.4.3”
Warning message:
“package ‘purrr’ was built under R version 4.4.3”
Warning message:
“package ‘readr’ was built under R version 4.4.3”
Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘patchwork’ was built under R version 4.4.3”
Spherical geometry (s2) switched off



## 1. Configuration
Every threshold that affects which pairs are used is set here and re-tested in Section 7. Paths can be overridden with environment variables (used for off-platform testing).

In [ ]:
DATA_STAMP <- Sys.getenv("VAL_DATA_STAMP", "20260513")
REF_DIR    <- Sys.getenv("VAL_REF_DIR", "/projects/my-private-bucket/reference")
FN <- list(
    eurasia = file.path(REF_DIR, paste0("eurasia_forest_structure_plots_agbd_", DATA_STAMP, ".gpkg")),
    nfi     = file.path(REF_DIR, paste0("nfi_plus_20220603_", DATA_STAMP, ".geojson")),
    miesner = file.path(REF_DIR, paste0("Miesner_plots_", DATA_STAMP, ".gpkg")),
    kolyma  = file.path(REF_DIR, paste0("KolymaRegion_all_", DATA_STAMP, ".gpkg"))
)
# Eurasia/NASA plot heights (Ht_med, Ht_max) are not in the AGBD file; join by `site`
EURASIA_HT_FN <- Sys.getenv("VAL_EURASIA_HT_FN",
    "/projects/my-public-bucket/databank/extract_from_points/eurasia_forest_structure_plots_smrytrees_20240124_s3_20241003_wGlobalCHM_s3_20250922.gpkg")
OUT_DIR <- Sys.getenv("VAL_OUT_DIR", file.path("/projects/my-private-bucket/validation_v3.1", format(Sys.Date(), "%Y%m%d")))
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

MAP_YEARS    <- 2016:2025
AGE_REF_YEAR <- 2020   # the age map describes stand age in 2020

# --- Stand-age screening ---
MATURE_AGE     <- 60   # [yr] stand age at field measurement above which height/AGBD change slowly enough to bridge long gaps
MAX_GAP_YOUNG  <- 5    # [yr] max |map year - field year| allowed for stands younger than MATURE_AGE (or of unknown age)
AGE_RULE       <- "mean"  # "mean": disturbed if age < years since obs; "conservative": also drop if age - 1 sd < years since obs
NONFOREST_HT   <- 2    # [m]     reference below this = non-forest plot (age map is not expected to have an age there)
NONFOREST_AGBD <- 5    # [Mg/ha]

# --- Post-2020 disturbance (not visible in the 2020 age map): abrupt, large drop in the mapped series ---
DROP_HT_M      <- 5    ; DROP_HT_FRAC   <- 0.5   # drop below the 2016–2020 median by > 5 m AND > 50 %
DROP_AGBD_MIN  <- 20   ; DROP_AGBD_FRAC <- 0.5   # drop below the 2016–2020 median by > 20 Mg/ha AND > 50 %
USE_DROP_FLAG  <- TRUE

# --- Reference definitions ---
HT_REF_MODE <- "top"    # "top": closest to canopy-top height (NFI site height, NASA max height, Miesner p98)
                        # "legacy": definitions used in earlier notebooks (NFI Lorey's height, NASA median height, Miesner p98)
KOLYMA_BIOMASS_TO_MGHA <- 0.01   # larch_biomass assumed g/m2 (median ~600) -> Mg/ha. VERIFY against Alexander et al. metadata.

# --- Groups excluded a priori (documented reasons) ---
EXCLUDE <- tribble(
    ~source,   ~group_name,           ~variable, ~reason,
    "NASA",    "Changbai Mountains",  "both",    "temperate mountain forest, outside the boreal domain",
    "NASA",    "Northernmost (AWI)",  "ht",      "same plots as Miesner (AWI); height kept from Miesner only",
    "NASA",    "Amur River",          "agbd",    "timber volume only, no tree-level biomass",
    "NASA",    "Russian Far East",    "agbd",    "Alnus likely recorded as Betula; Alnus has no allometry so plot AGBD is underestimated"
)

N_BOOT <- 1000
set.seed(42)

# Colors: reference categorical slots 1-4 (sources) and reserved status colors (screening state)
PAL_SOURCE <- c("NFI" = "#2a78d6", "NASA" = "#eb6834", "Miesner" = "#1baf7a", "Kolyma" = "#eda100")
PAL_AGE    <- c("undisturbed" = "#0ca30c", "marginal" = "#fab219",
                "disturbed after obs" = "#d03b3b", "no mapped age" = "#8c8c8c")
PAL_KEEP   <- c("retained" = "#2a78d6", "excluded" = "#eb6834")
PAL_MATURE <- c("mature at obs" = "#2a78d6", "young at obs" = "#eb6834", "unknown age" = "#8c8c8c")
theme_set(theme_bw(base_size = 11) + theme(panel.grid.minor = element_blank(), strip.background = element_rect(fill = "grey95")))

save_fig <- function(p, name, w, h) {
    ggsave(file.path(OUT_DIR, paste0(name, ".png")), p, width = w, height = h, dpi = 250, bg = "white")
    p
}

## 2. Helpers
**Metrics are computed against the 1:1 line.** Earlier notebooks (`lm_eqn`, `lm_eqn_boreal_ht`) reported the OLS *intercept* as "bias" and the OLS *residual* SE as "RMSE"; those describe the fitted line, not map error. Here:
* `bias` = mean(map − ref), `RMSD` = sqrt(mean((map − ref)²)), `MAE`, relative versions as % of mean reference;
* `r2` = squared Pearson correlation; OLS `slope`/`intercept` are kept only to describe attenuation.

**Confidence intervals** come from a *cluster* bootstrap that resamples plot groups (source × group × field year), because plots within a group share location, crew, protocol and often the same map pixels' neighbourhoods. A plot-level bootstrap would give falsely narrow intervals.

In [ ]:
pick_col <- function(df, patterns) {
    nm <- names(df)
    for (p in patterns) {
        hit <- nm[str_detect(nm, regex(p, ignore_case = TRUE))]
        if (length(hit) > 0) return(hit[1])
    }
    NA_character_
}
num <- function(x) suppressWarnings(as.numeric(x))
clean_ref <- function(x) { x <- num(x); x[!is.finite(x) | x <= -999] <- NA; x }

val_metrics <- function(pred, ref) {
    ok <- is.finite(pred) & is.finite(ref)
    pred <- pred[ok]; ref <- ref[ok]; n <- length(ref)
    if (n < 3) return(tibble(n = n))
    d <- pred - ref
    fit <- tryCatch(coef(lm(pred ~ ref)), error = function(e) c(NA, NA))
    tibble(n = n,
           mean_ref = mean(ref),
           bias = mean(d), rbias = 100 * mean(d) / mean(ref),
           rmsd = sqrt(mean(d^2)), rrmsd = 100 * sqrt(mean(d^2)) / mean(ref),
           mae = mean(abs(d)),
           r2 = if (sd(pred) > 0 && sd(ref) > 0) cor(pred, ref)^2 else NA_real_,
           slope = unname(fit[2]), intercept = unname(fit[1]))
}

boot_ci <- function(df, B = N_BOOT) {
    cl <- unique(df$cluster_id)
    if (length(cl) < 5) return(tibble(bias_lo = NA, bias_hi = NA, rmsd_lo = NA, rmsd_hi = NA, n_clusters = length(cl)))
    idx <- split(seq_len(nrow(df)), df$cluster_id)
    bs <- replicate(B, {
        rows <- unlist(idx[sample(names(idx), length(idx), replace = TRUE)], use.names = FALSE)
        d <- df$pred[rows] - df$ref[rows]
        c(mean(d), sqrt(mean(d^2)))
    })
    q <- apply(bs, 1, quantile, c(0.025, 0.975), na.rm = TRUE)
    tibble(bias_lo = q[1, 1], bias_hi = q[2, 1], rmsd_lo = q[1, 2], rmsd_hi = q[2, 2], n_clusters = length(cl))
}

summarise_val <- function(df, ..., ci = TRUE) {
    df %>%
        group_by(...) %>%
        group_modify(function(d, k) {
            m <- val_metrics(d$pred, d$ref)
            if (ci && nrow(d) >= 3) m <- bind_cols(m, boot_ci(d))
            m %>% mutate(n_plots = n_distinct(d$uid))
        }) %>%
        ungroup()
}

fmt_lab <- function(m, unit) {
    sprintf("n = %d\nbias = %+.1f %s\nRMSD = %.1f %s (%.0f%%)\nR² = %.2f",
            m$n, m$bias, unit, m$rmsd, unit, m$rrmsd, m$r2)
}

## 3. Load and harmonize reference plots
Each source is reduced to a common schema: `uid, source, group_name, year_obs, ref_ht, ref_ht_def, ref_agbd, ref_agbd_def, field_age`, plus the extracted map values.

| source | plots | height reference (`top` / `legacy`) | AGBD reference |
|---|---|---|---|
| Canadian NFI | ground plots, 400–500 m² | site height (dominant trees) / Lorey's height | live large + small trees (`plotbio_lgtr_live + plotbio_smtr_live`) |
| NASA Eurasia | GLAS-footprint plots, 314–707 m² | max tree height / median tree height | `agbd_v2_mg_ha` (tree allometry) |
| Miesner (AWI) | 707 m² circular | 98th percentile tree height (both modes); 0 for treeless plots | – |
| Kolyma | larch plots | – | `larch_biomass` × 0.01 (g/m² → Mg/ha, **to verify**) |

**NFI AGBD correction.** `plotbio_lgtr_live` already equals stem wood + bark + branches + foliage (checked: difference ≤ 0.02 Mg/ha on all 987 plots). The earlier `total_plotbio` (sum of every `plotbio*` column) double counts those components and adds dead trees, woody debris, stumps, shrubs and bryophytes: a median 2.3× inflation over live tree AGBD.

In [ ]:
read_any <- function(fn) {
    if (!file.exists(fn)) { message("MISSING: ", fn); return(NULL) }
    x <- suppressWarnings(st_read(fn, quiet = TRUE))
    if (is.na(st_crs(x))) st_crs(x) <- 4326
    st_transform(x, 4326)
}
raw <- map(FN, read_any)
walk2(names(raw), raw, ~ if (!is.null(.y)) message(sprintf("%-8s %5d rows  %3d cols", .x, nrow(.y), ncol(.y))))

In [ ]:
# Map columns -> long table (one row per plot x map year). Handles current (`value_Ht_H30_YYYY_v3.1_multiyr_mean_ht`)
# and legacy (`value_ht_H30_YYYY_v3.1_multiyr`) names.
MAP_RE <- "^value_(ht|agb)_h30_(\\d{4})_v3\\.1_multiyr(?:_(mean|std)_(?:ht|agbd))?$"

map_long <- function(df) {
    cols <- names(df)[str_detect(tolower(names(df)), MAP_RE)]
    df %>%
        st_drop_geometry() %>%
        select(uid, all_of(cols)) %>%
        pivot_longer(-uid, names_to = "col", values_to = "value") %>%
        mutate(m = str_match(tolower(col), MAP_RE),
               variable = recode(m[, 2], ht = "ht", agb = "agbd"),
               map_year = as.integer(m[, 3]),
               stat = coalesce(m[, 4], "mean"),
               value = num(value),
               value = if_else(is.finite(value) & value >= 0, value, NA_real_)) %>%
        select(uid, variable, map_year, stat, value) %>%
        pivot_wider(names_from = stat, values_from = value) %>%
        rename(pred = mean, pred_sd = any_of("std"))
}

aux_cols <- function(df) {
    tibble(
        age_map    = clean_ref(df[[pick_col(df, "^value_CACC_2020_v3\\.1_multiyr_age_mean$")]] %||% NA),
        age_map_sd = clean_ref(df[[pick_col(df, "^value_CACC_2020_v3\\.1_multiyr_age_std$")]] %||% NA),
        agb_trend_slope = clean_ref(df[[pick_col(df, "multiyr_2020-2025_trendslope$")]] %||% NA),
        agb_trend_p     = clean_ref(df[[pick_col(df, "multiyr_2020-2025_kendall_pvalue$")]] %||% NA))
}
`%||%` <- function(a, b) if (is.null(a)) b else a

In [ ]:
harmonize_nfi <- function(x) {
    if (is.null(x)) return(NULL)
    date_col <- pick_col(x, c("^meas_date_x$", "^meas_date$"))
    lorey <- clean_ref(x$lorey_height); site <- clean_ref(x$site_height)
    lg <- clean_ref(x$plotbio_lgtr_live); sm <- clean_ref(x$plotbio_smtr_live)
    tibble(source = "NFI",
           group_name = x$ecozone,
           year_obs = as.integer(str_sub(as.character(x[[date_col]]), 1, 4)),
           ref_ht_top = coalesce(site, lorey), ref_ht_legacy = coalesce(lorey, site),
           ref_ht_def_top = if_else(is.na(site), "Lorey's height (no site height)", "site height"),
           ref_ht_def_legacy = if_else(is.na(lorey), "site height (no Lorey)", "Lorey's height"),
           ref_agbd = if_else(is.na(lg), NA_real_, lg + coalesce(sm, 0)),
           ref_agbd_def = if_else(is.na(sm), "live large trees (small trees not measured)", "live large + small trees"),
           field_age = clean_ref(x$site_age),
           plot_id = as.character(x$nfi_plot)) %>%
        bind_cols(aux_cols(x)) %>% st_sf(geometry = st_geometry(x))
}

harmonize_nasa <- function(x, ht_fn) {
    if (is.null(x)) return(NULL)
    ht <- if (file.exists(ht_fn)) {
        st_read(ht_fn, quiet = TRUE) %>% st_drop_geometry() %>%
            transmute(site = as.character(site), Ht_med = clean_ref(Ht_med), Ht_max = clean_ref(Ht_max))
    } else { message("MISSING NASA height file: ", ht_fn); tibble(site = character(), Ht_med = numeric(), Ht_max = numeric()) }
    stopifnot(!any(duplicated(ht$site)))
    x$site <- as.character(x$site)
    j <- left_join(st_drop_geometry(x) %>% select(site), ht, by = "site")
    message(sprintf("NASA: %d / %d plots matched a height record by site", sum(!is.na(j$Ht_med) | !is.na(j$Ht_max)), nrow(x)))
    tibble(source = "NASA",
           group_name = x$group_name,
           year_obs = as.integer(x$year),
           ref_ht_top = coalesce(j$Ht_max, j$Ht_med), ref_ht_legacy = j$Ht_med,
           ref_ht_def_top = if_else(is.na(j$Ht_max), "median tree height (no max)", "max tree height"),
           ref_ht_def_legacy = "median tree height",
           ref_agbd = clean_ref(x$agbd_v2_mg_ha),
           ref_agbd_def = "agbd_v2 (tree allometry)",
           field_age = NA_real_,
           plot_id = x$site) %>%
        bind_cols(aux_cols(x)) %>% st_sf(geometry = st_geometry(x))
}

harmonize_miesner <- function(x) {
    if (is.null(x)) return(NULL)
    camp <- x[[pick_col(x, "^campaign$")]]
    p98 <- clean_ref(x[[pick_col(x, c("98th"))]])
    ntree <- clean_ref(x[[pick_col(x, c("^Trees \\[#\\]$", "^Trees\\.\\.\\.\\.$"))]])
    ref <- if_else(!is.na(ntree) & ntree == 0, 0, p98)
    tibble(source = "Miesner",
           group_name = str_remove(camp, "^[^_]+_\\d{4}_"),
           year_obs = as.integer(str_extract(camp, "(?<=_)\\d{4}(?=_)")),
           ref_ht_top = ref, ref_ht_legacy = ref,
           ref_ht_def_top = if_else(!is.na(ntree) & ntree == 0, "treeless plot (0 m)", "98th pct tree height"),
           ref_ht_def_legacy = ref_ht_def_top,
           ref_agbd = NA_real_, ref_agbd_def = NA_character_, field_age = NA_real_,
           plot_id = as.character(x[[pick_col(x, c("^event$", "^X\\.Event$", "^site$"))]])) %>%
        bind_cols(aux_cols(x)) %>% st_sf(geometry = st_geometry(x))
}

harmonize_kolyma <- function(x) {
    if (is.null(x)) return(NULL)
    tibble(source = "Kolyma",
           group_name = paste0("Kolyma: ", x$project),
           year_obs = as.integer(num(x$year)),
           ref_ht_top = NA_real_, ref_ht_legacy = NA_real_, ref_ht_def_top = NA_character_, ref_ht_def_legacy = NA_character_,
           ref_agbd = clean_ref(x$larch_biomass) * KOLYMA_BIOMASS_TO_MGHA,
           ref_agbd_def = "larch biomass (g/m2 x 0.01)",
           field_age = NA_real_,
           plot_id = paste(x$project, x$site, x$plot, sep = "_")) %>%
        bind_cols(aux_cols(x)) %>% st_sf(geometry = st_geometry(x))
}

parts <- list(NFI = harmonize_nfi(raw$nfi), NASA = harmonize_nasa(raw$eurasia, EURASIA_HT_FN),
              Miesner = harmonize_miesner(raw$miesner), Kolyma = harmonize_kolyma(raw$kolyma))
src_raw <- list(NFI = raw$nfi, NASA = raw$eurasia, Miesner = raw$miesner, Kolyma = raw$kolyma)
parts <- compact(parts)

plots <- imap(parts, function(p, nm) {
    p$uid <- paste0(nm, "_", seq_len(nrow(p)))
    p
}) %>% bind_rows()

pred_long <- imap(parts, function(p, nm) {
    x <- src_raw[[nm]]
    x$uid <- paste0(nm, "_", seq_len(nrow(x)))
    map_long(x)
}) %>% bind_rows()

In [ ]:
# Regions. Fixes vs. earlier notebooks: 'Bakhta River' was misspelled 'Bahkta River' in the height notebooks (so it was
# labelled permafrost boreal); southern-taiga groups Taseevo, Mana, Sisim, South of Lake Baikal were not assigned.
BOREAL_EA    <- c("West Siberian Plains", "Bakhta River", "Tunguska River", "Tahe", "Russian Far East",
                  "Taseevo", "Mana River", "Sisim River", "South of Lake Baikal", "Amur River")
TEMPERATE_NA <- c("Atlantic Maritime", "Pacific Maritime", "Mixedwood Plains")

plots <- plots %>%
    mutate(continent = if_else(source == "NFI", "North America", "Eurasia"),
           region = case_when(source == "NFI" & group_name %in% TEMPERATE_NA ~ "Temperate",
                              source == "NFI" ~ "Boreal",
                              group_name %in% BOREAL_EA ~ "Boreal",
                              TRUE ~ "Permafrost boreal"),
           in_boreal_domain = region != "Temperate",
           ref_ht = if (HT_REF_MODE == "top") ref_ht_top else ref_ht_legacy,
           ref_ht_def = if (HT_REF_MODE == "top") ref_ht_def_top else ref_ht_def_legacy,
           cluster_id = paste(source, group_name, year_obs, sep = " | "))

excl_ht   <- EXCLUDE %>% filter(variable %in% c("ht", "both"))
excl_agbd <- EXCLUDE %>% filter(variable %in% c("agbd", "both"))
plots <- plots %>%
    mutate(excl_ht   = paste(source, group_name) %in% paste(excl_ht$source, excl_ht$group_name),
           excl_agbd = paste(source, group_name) %in% paste(excl_agbd$source, excl_agbd$group_name))

### 3.1 Reference inventory

In [ ]:
inventory <- plots %>% st_drop_geometry() %>%
    group_by(source, region, group_name) %>%
    summarise(n_plots = n(),
              years = if (all(is.na(year_obs))) NA_character_ else paste(unique(range(year_obs, na.rm = TRUE)), collapse = "–"),
              n_ht = sum(!is.na(ref_ht) & !excl_ht), n_agbd = sum(!is.na(ref_agbd) & !excl_agbd),
              median_ref_ht = median(ref_ht, na.rm = TRUE), median_ref_agbd = median(ref_agbd, na.rm = TRUE),
              pct_with_age = 100 * mean(!is.na(age_map) & age_map > 0)) %>%
    ungroup() %>% arrange(source, region, group_name)
write_csv(inventory, file.path(OUT_DIR, "T1_reference_inventory.csv"))
inventory %>% print(n = Inf)

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 4.5, repr.plot.res = 200)
p_years <- plots %>% st_drop_geometry() %>% filter(!is.na(year_obs)) %>%
    ggplot(aes(year_obs, fill = source)) +
    annotate("rect", xmin = min(MAP_YEARS) - 0.5, xmax = max(MAP_YEARS) + 0.5, ymin = -Inf, ymax = Inf, fill = "grey88") +
    annotate("text", x = mean(MAP_YEARS), y = Inf, vjust = 1.5, label = "v3.1 map years", size = 3.2, colour = "grey30") +
    geom_bar(width = 0.8) +
    scale_fill_manual(values = PAL_SOURCE, name = NULL) +
    labs(x = "Year of field measurement", y = "Plots", title = "When the reference plots were measured") +
    theme(legend.position = "top")
save_fig(p_years, "F01_field_years", 9, 3.5)

## 4. Stand-age screening
### 4.1 Rule
For a plot measured in year *t₀*, the 2020 age map gives the stand age *A*₂₀₂₀ (ensemble mean ± sd).

* **Disturbed after field measurement** if *A*₂₀₂₀ < 2020 − *t₀*: the stand in 2020 is younger than the time elapsed since the plot was measured, so it was replaced after measurement and the plot no longer describes the pixel. This is the rule used in `Review_maps_v3.1_trends_VHR_CHM.ipynb`.
* **Marginal** if the rule passes on the mean but fails at *A*₂₀₂₀ − 1 sd. Kept by default (`AGE_RULE = "mean"`), dropped under `"conservative"`.
* **No mapped age**: non-forest in the age map, or outside its mask. Kept only where the plot is also non-forest (height < 2 m / AGBD < 5 Mg/ha). A forested plot without a mapped age has an inconsistent land-cover designation, so it is not trusted.
* **Age at measurement** *A*₀ = *A*₂₀₂₀ − (2020 − *t₀*). Stands with *A*₀ ≥ `MATURE_AGE` change slowly enough to bridge long gaps. Younger (or unknown-age, forested) stands are used only within ±`MAX_GAP_YOUNG` years of the map year.
* **After 2020** the age map is blind. A plot × year pair is dropped from the first map year (2021–2025) in which the mapped series falls by more than 5 m and 50 % (height) or 20 Mg/ha and 50 % (AGBD) below its 2016–2020 median. This uses the product to screen itself. It removes few pairs (reported below) and results are shown with and without it.

**Tiers**: `all` (no screening, like earlier notebooks) → `age-screened` (no disturbance after measurement, consistent land cover, no post-2020 drop) → `strict` (age-screened and growth-stable across the gap).

**Matched year**: each plot is compared to the map year closest to its field year (`clamp(t₀, 2016, 2025)`). All-year pairs are used only for the gap and time-series analyses.

In [ ]:
plots <- plots %>%
    mutate(yrs_since_obs = AGE_REF_YEAR - year_obs,
           age_ok = !is.na(age_map) & age_map > 0,
           age_class = case_when(!age_ok ~ "no mapped age",
                                 age_map < yrs_since_obs ~ "disturbed after obs",
                                 !is.na(age_map_sd) & (age_map - age_map_sd) < yrs_since_obs ~ "marginal",
                                 TRUE ~ "undisturbed"),
           age_class = factor(age_class, levels = names(PAL_AGE)),
           age_at_obs = if_else(age_ok, age_map - yrs_since_obs, NA_real_),
           maturity = case_when(!age_ok ~ "unknown age",
                                age_at_obs >= MATURE_AGE ~ "mature at obs",
                                TRUE ~ "young at obs"),
           nonforest_ht = !is.na(ref_ht) & ref_ht < NONFOREST_HT,
           nonforest_agbd = !is.na(ref_agbd) & ref_agbd < NONFOREST_AGBD)

plots %>% st_drop_geometry() %>% count(source, age_class) %>% pivot_wider(names_from = age_class, values_from = n, values_fill = 0)

In [ ]:
# Post-2020 abrupt-drop flag from the mapped series itself
drop_flags <- pred_long %>%
    group_by(uid, variable) %>%
    summarise(base = median(pred[map_year <= 2020], na.rm = TRUE),
              drop_year = {
                  yy <- map_year[map_year > 2020 & !is.na(pred)]
                  pp <- pred[map_year > 2020 & !is.na(pred)]
                  b <- median(pred[map_year <= 2020], na.rm = TRUE)
                  hit <- if (first(variable) == "ht") pp < b - DROP_HT_M & pp < (1 - DROP_HT_FRAC) * b
                         else pp < b - DROP_AGBD_MIN & pp < (1 - DROP_AGBD_FRAC) * b
                  if (any(hit, na.rm = TRUE)) min(yy[which(hit)]) else NA_integer_
              }) %>%
    group_by(uid) %>%
    summarise(drop_year = if (all(is.na(drop_year))) NA_integer_ else as.integer(min(drop_year, na.rm = TRUE)))
plots <- plots %>% select(-any_of("drop_year")) %>% left_join(drop_flags, by = "uid")
message(sprintf("Plots with a post-2020 drop in the mapped series: %d", sum(!is.na(plots$drop_year))))

In [ ]:
# Build validation pairs (plot x variable x map year) and apply the tiers
make_pairs <- function(plots, pred_long, mature_age = MATURE_AGE, max_gap_young = MAX_GAP_YOUNG,
                       age_rule = AGE_RULE, use_drop = USE_DROP_FLAG) {
    pl <- plots %>% st_drop_geometry() %>%
        mutate(maturity = case_when(!age_ok ~ "unknown age", age_at_obs >= mature_age ~ "mature at obs", TRUE ~ "young at obs"))
    ref_long <- bind_rows(
        pl %>% filter(!excl_ht)   %>% transmute(uid, variable = "ht",   ref = ref_ht,   nonforest = nonforest_ht),
        pl %>% filter(!excl_agbd) %>% transmute(uid, variable = "agbd", ref = ref_agbd, nonforest = nonforest_agbd)) %>%
        filter(!is.na(ref))
    ref_long %>%
        inner_join(pred_long %>% filter(!is.na(pred)), by = c("uid", "variable")) %>%
        left_join(pl %>% select(uid, source, group_name, region, continent, in_boreal_domain, cluster_id, year_obs,
                                age_class, age_map, age_map_sd, age_at_obs, maturity, drop_year, yrs_since_obs, field_age),
                  by = "uid") %>%
        filter(!is.na(year_obs)) %>%
        mutate(gap = map_year - year_obs,
               matched_year = pmin(pmax(year_obs, min(MAP_YEARS)), max(MAP_YEARS)),
               is_matched = map_year == matched_year,
               age_pass = case_when(age_class == "undisturbed" ~ TRUE,
                                    age_class == "marginal" ~ age_rule == "mean",
                                    age_class == "no mapped age" ~ nonforest,
                                    TRUE ~ FALSE),
               post_drop = use_drop & !is.na(drop_year) & map_year >= drop_year,
               screened = age_pass & !post_drop,
               growth_stable = maturity == "mature at obs" | abs(gap) <= max_gap_young |
                               (age_class == "no mapped age" & nonforest),
               strict = screened & growth_stable,
               exclusion = case_when(age_class == "disturbed after obs" ~ "disturbed after obs (age map)",
                                     age_class == "marginal" & !age_pass ~ "marginal age (conservative rule)",
                                     age_class == "no mapped age" & !nonforest ~ "forest plot, no mapped age",
                                     post_drop ~ "post-2020 drop in map series",
                                     !growth_stable ~ "young stand, gap too long",
                                     TRUE ~ "retained (strict)"))
}
pairs <- make_pairs(plots, pred_long)
matched <- pairs %>% filter(is_matched)

tier_long <- function(d) bind_rows(d %>% mutate(tier = "all"),
                                   d %>% filter(screened) %>% mutate(tier = "age-screened"),
                                   d %>% filter(strict) %>% mutate(tier = "strict")) %>%
    mutate(tier = factor(tier, levels = c("all", "age-screened", "strict")))

### 4.2 What the screen removes
Reasons are applied in the order listed, so each excluded matched pair is counted once.

In [ ]:
screen_flow <- matched %>%
    count(variable, source, exclusion) %>%
    pivot_wider(names_from = exclusion, values_from = n, values_fill = 0) %>%
    arrange(variable, source)
write_csv(screen_flow, file.path(OUT_DIR, "T2_screening_flow_matched_year.csv"))
screen_flow

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)
p_rule <- plots %>% st_drop_geometry() %>% filter(!is.na(year_obs)) %>%
    mutate(age_plot = if_else(age_ok, age_map, -8)) %>%
    ggplot(aes(yrs_since_obs, age_plot, colour = age_class)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = "grey30") +
    geom_jitter(width = 0.25, height = 0, size = 1.6, alpha = 0.75) +
    scale_colour_manual(values = PAL_AGE, name = NULL, drop = FALSE) +
    scale_y_continuous(breaks = c(-8, seq(0, 150, 25)), labels = c("none", seq(0, 125, 25), "\u2265150"),
                       limits = c(-10, 150), oob = scales::squish) +
    facet_wrap(~source, nrow = 1) +
    labs(x = "Years from field measurement to 2020", y = "Mapped stand age in 2020 [yr]",
         title = "Stand-age screening",
         subtitle = "Below the dashed 1:1 line the stand is younger than the time since measurement: it was replaced after the plot was measured. Ages \u2265150 yr drawn at the top.") +
    theme(legend.position = "top")
save_fig(p_rule, "F02_age_screening_rule", 12, 4.5)

In [ ]:
# Groups flagged in earlier manual review (Google Earth): does the age screen agree?
manual_check <- plots %>% st_drop_geometry() %>%
    filter(group_name %in% c("West Siberian Plains", "Tunguska River")) %>%
    count(group_name, year_obs, age_class) %>%
    pivot_wider(names_from = age_class, values_from = n, values_fill = 0)
write_csv(manual_check, file.path(OUT_DIR, "T10_manual_review_groups_age_class.csv"))
manual_check

### 4.3 Is the age map credible at plot scale? NFI field age vs mapped age
NFI plots record a field stand age (`site_age`, from cored site trees). Expected 2020 age = `site_age + (2020 − t₀)`. Agreement supports the screen; plots where the map is far *younger* than expected are where the screen fires (true disturbance or age-map error, which cannot be told apart without imagery).

In [ ]:
nfi_age <- plots %>% st_drop_geometry() %>% filter(source == "NFI", !is.na(field_age), age_ok) %>%
    mutate(expected_2020 = field_age + yrs_since_obs)
if (nrow(nfi_age) > 0) {
    age_cmp <- val_metrics(nfi_age$age_map, nfi_age$expected_2020) %>%
        mutate(spearman = cor(nfi_age$age_map, nfi_age$expected_2020, method = "spearman"),
               pct_map_younger_than_elapsed = 100 * mean(nfi_age$age_map < nfi_age$yrs_since_obs),
               pct_field_young_at_obs = 100 * mean(nfi_age$field_age < MATURE_AGE),
               pct_maturity_agree = 100 * mean((nfi_age$field_age >= MATURE_AGE) == (nfi_age$age_at_obs >= MATURE_AGE)))
    write_csv(age_cmp, file.path(OUT_DIR, "T8_nfi_field_age_vs_map_age.csv"))
    print(age_cmp)
    options(repr.plot.width = 6, repr.plot.height = 5.5)
    p_age <- ggplot(nfi_age, aes(expected_2020, age_map, colour = age_class)) +
        geom_abline(linetype = "dashed", colour = "grey30") +
        geom_point(size = 1.6, alpha = 0.75) +
        scale_colour_manual(values = PAL_AGE, name = NULL, drop = FALSE) +
        coord_equal(xlim = c(0, 300), ylim = c(0, 300)) +
        annotate("label", x = 5, y = 295, hjust = 0, vjust = 1, size = 3, label.size = 0,
                 label = sprintf("n = %d\nbias = %+.0f yr\nRMSD = %.0f yr\nSpearman = %.2f",
                                 age_cmp$n, age_cmp$bias, age_cmp$rmsd, age_cmp$spearman)) +
        labs(x = "Field stand age projected to 2020 [yr]", y = "Mapped stand age 2020 [yr]",
             title = "NFI: field vs mapped stand age") +
        theme(legend.position = "bottom")
    save_fig(p_age, "F03_nfi_field_vs_map_age", 6, 5.5)
}

In [ ]:
# Map of plots coloured by age class
options(repr.plot.width = 14, repr.plot.height = 5)
world <- tryCatch(rnaturalearth::ne_countries(scale = "medium", returnclass = "sf"), error = function(e) NULL)
map_panel <- function(cont, xl, yl) {
    d <- plots %>% filter(continent == cont)
    p <- ggplot()
    if (!is.null(world)) p <- p + geom_sf(data = world, fill = "grey92", colour = "grey70", linewidth = 0.2)
    p + geom_sf(data = d, aes(colour = age_class), size = 1.4, alpha = 0.8) +
        scale_colour_manual(values = PAL_AGE, name = NULL, drop = FALSE) +
        coord_sf(xlim = xl, ylim = yl, expand = FALSE) +
        labs(title = cont)
}
p_map <- (map_panel("North America", c(-145, -50), c(41, 72)) + map_panel("Eurasia", c(60, 180), c(40, 75))) +
    plot_layout(guides = "collect") & theme(legend.position = "bottom")
save_fig(p_map, "F04_map_plots_age_class", 14, 5)

## 5. Height validation
### 5.1 Matched-year pairs: what screening retains and removes
Labels: metrics for **retained (strict)** pairs (bottom) vs **all** pairs (top). Boreal domain only; temperate NFI ecozones are reported separately in the tables.

In [ ]:
val_panel <- function(var, unit, lim) {
    d <- matched %>% filter(variable == var, in_boreal_domain) %>%
        mutate(keep = factor(if_else(strict, "retained", "excluded"), levels = names(PAL_KEEP)))
    lab_tbl <- function(dd, prefix) dd %>% group_by(source) %>% group_modify(~ val_metrics(.x$pred, .x$ref)) %>% ungroup() %>%
        filter(n >= 3) %>% mutate(lab = paste0(prefix, "\n", fmt_lab(pick(everything()), unit)))
    m_all <- lab_tbl(d, "all")
    m_str <- lab_tbl(filter(d, strict), "strict")
    ggplot(d, aes(ref, pred)) +
        geom_abline(linetype = "dashed", colour = "grey30") +
        geom_point(aes(colour = keep), size = 1.3, alpha = 0.6) +
        geom_smooth(data = filter(d, strict), method = "lm", formula = y ~ x, se = FALSE, colour = "#2a78d6", linewidth = 0.7) +
        geom_label(data = m_all, aes(x = 0, y = lim, label = lab), hjust = 0, vjust = 1, size = 2.5, label.size = 0, alpha = 0.8) +
        geom_label(data = m_str, aes(x = lim, y = 0, label = lab), hjust = 1, vjust = 0, size = 2.5, label.size = 0, alpha = 0.8, colour = "#184f95") +
        scale_colour_manual(values = PAL_KEEP, name = NULL) +
        coord_equal(xlim = c(0, lim), ylim = c(0, lim)) +
        facet_wrap(~source, nrow = 1) +
        theme(legend.position = "top")
}
options(repr.plot.width = 12, repr.plot.height = 5)
p_ht <- val_panel("ht", "m", 35) +
    labs(x = sprintf("Field reference height [m] (%s definitions)", HT_REF_MODE), y = "v3.1 mapped height, matched year [m]",
         title = "Vegetation height v3.1 vs field plots (matched map year)")
save_fig(p_ht, "F05_height_matched_retained_vs_excluded", 12, 5)

### 5.2 Headline metrics by tier

In [ ]:
metrics_by <- function(d, ...) d %>% tier_long() %>% summarise_val(variable, tier, ...)
m_headline <- bind_rows(
    matched %>% filter(in_boreal_domain) %>% metrics_by() %>% mutate(group = "boreal domain, pooled"),
    matched %>% filter(in_boreal_domain) %>% metrics_by(source) %>% mutate(group = source),
    matched %>% metrics_by(region) %>% mutate(group = region)) %>%
    select(variable, group, tier, n, n_plots, n_clusters, mean_ref, bias, bias_lo, bias_hi, rbias, rmsd, rmsd_lo, rmsd_hi, rrmsd, mae, r2, slope, intercept)
write_csv(m_headline, file.path(OUT_DIR, "T3_metrics_headline.csv"))
m_headline %>% filter(variable == "ht") %>% mutate(across(where(is.numeric), ~ round(.x, 2))) %>% print(n = Inf)

In [ ]:
tier_plot <- function(var, unit) {
    d <- m_headline %>% filter(variable == var, !is.na(bias)) %>%
        mutate(group = factor(group, levels = rev(unique(group))))
    pb <- ggplot(d, aes(bias, group, colour = tier)) +
        geom_vline(xintercept = 0, colour = "grey50") +
        geom_pointrange(aes(xmin = bias_lo, xmax = bias_hi), position = position_dodge(width = 0.6), size = 0.3) +
        labs(x = paste0("Bias (map − field) [", unit, "]"), y = NULL)
    pr <- ggplot(d, aes(rmsd, group, colour = tier)) +
        geom_pointrange(aes(xmin = rmsd_lo, xmax = rmsd_hi), position = position_dodge(width = 0.6), size = 0.3) +
        labs(x = paste0("RMSD [", unit, "]"), y = NULL) + theme(axis.text.y = element_blank())
    (pb + pr) + plot_layout(guides = "collect") &
        scale_colour_manual(values = c("all" = "#8c8c8c", "age-screened" = "#86b6ef", "strict" = "#184f95"), name = NULL) &
        theme(legend.position = "top")
}
options(repr.plot.width = 11, repr.plot.height = 4.5)
p_ht_tiers <- tier_plot("ht", "m") + plot_annotation(title = "Height: effect of stand-age screening (95% cluster-bootstrap CI)")
save_fig(p_ht_tiers, "F06_height_metrics_by_tier", 11, 4.5)

### 5.3 Where the error lives: residuals vs reference height, and plot groups

In [ ]:
resid_panel <- function(var, unit, breaks) {
    matched %>% filter(variable == var, strict, in_boreal_domain) %>%
        mutate(ref_bin = cut(ref, breaks, include.lowest = TRUE, right = FALSE)) %>%
        ggplot(aes(ref_bin, pred - ref, fill = source)) +
        geom_hline(yintercept = 0, colour = "grey30") +
        geom_boxplot(outlier.size = 0.6, linewidth = 0.3, position = position_dodge2(preserve = "single")) +
        scale_fill_manual(values = PAL_SOURCE, name = NULL) +
        labs(x = paste0("Field reference [", unit, "]"), y = paste0("Map − field [", unit, "]")) +
        theme(legend.position = "top")
}
options(repr.plot.width = 10, repr.plot.height = 4)
p_ht_resid <- resid_panel("ht", "m", c(0, 2, 5, 10, 15, 20, 25, 50)) + labs(title = "Height residuals by reference height (strict pairs)")
save_fig(p_ht_resid, "F07_height_residual_by_ref", 10, 4)

In [ ]:
group_metrics <- matched %>% filter(strict) %>%
    group_by(variable, source, region, group_name) %>%
    group_modify(~ val_metrics(.x$pred, .x$ref) %>% mutate(years = paste(sort(unique(.x$year_obs)), collapse = ","))) %>%
    ungroup() %>% filter(n >= 5)
write_csv(group_metrics, file.path(OUT_DIR, "T5_metrics_by_plot_group_strict.csv"))

group_plot <- function(var, unit) {
    group_metrics %>% filter(variable == var) %>%
        mutate(group_name = reorder(group_name, bias)) %>%
        ggplot(aes(bias, group_name, colour = source)) +
        geom_vline(xintercept = 0, colour = "grey50") +
        geom_errorbarh(aes(xmin = bias - rmsd, xmax = bias + rmsd), height = 0, alpha = 0.5) +
        geom_point(aes(size = n)) +
        scale_colour_manual(values = PAL_SOURCE, name = NULL) +
        scale_size_area(max_size = 4, name = "pairs") +
        facet_grid(region ~ ., scales = "free_y", space = "free_y") +
        labs(x = paste0("Bias, bar = ± RMSD [", unit, "]"), y = NULL)
}
options(repr.plot.width = 8, repr.plot.height = 8)
p_ht_groups <- group_plot("ht", "m") + labs(title = "Height by plot group (strict, ≥5 pairs)")
save_fig(p_ht_groups, "F08_height_by_group", 8, 8)

## 6. AGBD validation
Same structure as height. NASA Amur River and Russian Far East are excluded a priori (see Config), and Kolyma AGBD carries a unit assumption.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)
p_agb <- val_panel("agbd", "Mg/ha", 250) +
    labs(x = "Field reference AGBD [Mg/ha]", y = "v3.1 mapped AGBD, matched year [Mg/ha]",
         title = "AGBD v3.1 vs field plots (matched map year)")
save_fig(p_agb, "F09_agbd_matched_retained_vs_excluded", 12, 5)

In [ ]:
m_headline %>% filter(variable == "agbd") %>% mutate(across(where(is.numeric), ~ round(.x, 2))) %>% print(n = Inf)
options(repr.plot.width = 11, repr.plot.height = 4.5)
p_agb_tiers <- tier_plot("agbd", "Mg/ha") + plot_annotation(title = "AGBD: effect of stand-age screening (95% cluster-bootstrap CI)")
save_fig(p_agb_tiers, "F10_agbd_metrics_by_tier", 11, 4.5)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)
p_agb_resid <- resid_panel("agbd", "Mg/ha", c(0, 5, 25, 50, 100, 150, 200, 1000)) + labs(title = "AGBD residuals by reference AGBD (strict pairs)")
save_fig(p_agb_resid, "F11_agbd_residual_by_ref", 10, 4)
options(repr.plot.width = 8, repr.plot.height = 8)
p_agb_groups <- group_plot("agbd", "Mg/ha") + labs(title = "AGBD by plot group (strict, ≥5 pairs)")
save_fig(p_agb_groups, "F12_agbd_by_group", 8, 8)

## 7. Time-series consistency
### 7.1 Bias vs time gap
Every plot × map year pair that passes the age screen. If stands were static, bias would be flat across the gap. A slope in young stands but not mature ones is the growth signal that motivates the `strict` maturity rule.

In [ ]:
gap_trend <- pairs %>% filter(screened, in_boreal_domain) %>%
    group_by(variable, maturity) %>%
    summarise(n = n(), slope_per_yr = if (n_distinct(gap) > 2) coef(lm(I(pred - ref) ~ gap))[2] else NA_real_)
write_csv(gap_trend, file.path(OUT_DIR, "T6_bias_trend_vs_gap.csv"))
print(gap_trend)

gap_panel <- function(var, unit, ylim) {
    pairs %>% filter(variable == var, screened, in_boreal_domain) %>%
        mutate(gap_bin = cut(gap, c(-Inf, -1, 2, 5, 10, 15, 20, Inf), labels = c("<0", "0–2", "3–5", "6–10", "11–15", "16–20", ">20"))) %>%
        ggplot(aes(gap_bin, pred - ref, fill = maturity)) +
        geom_hline(yintercept = 0, colour = "grey30") +
        geom_boxplot(outlier.shape = NA, linewidth = 0.3, position = position_dodge2(preserve = "single")) +
        coord_cartesian(ylim = ylim) +
        scale_fill_manual(values = PAL_MATURE, name = NULL) +
        labs(x = "Map year − field year [yr]", y = paste0("Map − field [", unit, "]")) +
        theme(legend.position = "top")
}
options(repr.plot.width = 11, repr.plot.height = 4)
p_gap <- (gap_panel("ht", "m", c(-15, 15)) + labs(title = "Height") |
          gap_panel("agbd", "Mg/ha", c(-120, 120)) + labs(title = "AGBD")) +
    plot_layout(guides = "collect") & theme(legend.position = "top")
save_fig(p_gap, "F13_bias_vs_gap_by_maturity", 11, 4)

### 7.2 Accuracy per map year, fixed plot set
Each map year is compared to the **same** plots (strict, valid in all ten years), so year-to-year changes reflect the map, not a changing sample. A flat line means the time series can be compared across years without year-specific bias.

In [ ]:
fixed <- pairs %>% filter(strict, in_boreal_domain) %>%
    group_by(variable, uid) %>% filter(n_distinct(map_year) == length(MAP_YEARS)) %>% ungroup()
m_year <- fixed %>% summarise_val(variable, map_year)
write_csv(m_year, file.path(OUT_DIR, "T7_metrics_by_map_year_fixed_set.csv"))

year_panel <- function(var, unit) {
    d <- m_year %>% filter(variable == var)
    pb <- ggplot(d, aes(map_year, bias)) + geom_hline(yintercept = 0, colour = "grey50") +
        geom_ribbon(aes(ymin = bias_lo, ymax = bias_hi), fill = "#cde2fb") + geom_line(colour = "#2a78d6", linewidth = 0.7) +
        geom_point(colour = "#2a78d6", size = 2.2) + scale_x_continuous(breaks = MAP_YEARS) + labs(x = NULL, y = paste0("Bias [", unit, "]"))
    pr <- ggplot(d, aes(map_year, rmsd)) +
        geom_ribbon(aes(ymin = rmsd_lo, ymax = rmsd_hi), fill = "#cde2fb") + geom_line(colour = "#2a78d6", linewidth = 0.7) +
        geom_point(colour = "#2a78d6", size = 2.2) + scale_x_continuous(breaks = MAP_YEARS) + labs(x = NULL, y = paste0("RMSD [", unit, "]"))
    (pb / pr) + plot_annotation(subtitle = sprintf("%s: same %d plots in every year", c(ht = "Height", agbd = "AGBD")[[var]], n_distinct(fixed$uid[fixed$variable == var])))
}
options(repr.plot.width = 11, repr.plot.height = 5)
p_year <- (wrap_elements(year_panel("ht", "m")) | wrap_elements(year_panel("agbd", "Mg/ha"))) +
    plot_annotation(title = "Accuracy per map year on a fixed set of strict plots (95% cluster-bootstrap CI)")
save_fig(p_year, "F14_metrics_by_map_year", 11, 5)

### 7.3 Mapped trajectories by screening class
Disturbed-after-measurement plots should show low or falling mapped values. Undisturbed plots should be flat or slowly rising. This is an internal-consistency check between the age map and the structure time series.

In [ ]:
traj <- pred_long %>% inner_join(plots %>% st_drop_geometry() %>% select(uid, source, age_class, in_boreal_domain), by = "uid") %>%
    filter(in_boreal_domain, !is.na(pred))
traj_med <- traj %>% group_by(variable, age_class, map_year) %>% summarise(med = median(pred), q25 = quantile(pred, 0.25), q75 = quantile(pred, 0.75), n = n_distinct(uid))
options(repr.plot.width = 11, repr.plot.height = 4)
p_traj <- ggplot(traj_med, aes(map_year, med, colour = age_class, fill = age_class)) +
    geom_ribbon(aes(ymin = q25, ymax = q75), alpha = 0.15, colour = NA) +
    geom_line(linewidth = 0.8) + geom_point(size = 1.8) +
    scale_colour_manual(values = PAL_AGE, name = NULL, aesthetics = c("colour", "fill")) +
    facet_wrap(~variable, scales = "free_y", labeller = as_labeller(c(ht = "Height [m]", agbd = "AGBD [Mg/ha]"))) +
    labs(x = "Map year", y = "Median (IQR) mapped value at plots", title = "Mapped 2016–2025 trajectories at plots, by stand-age class") +
    theme(legend.position = "top")
save_fig(p_traj, "F15_trajectories_by_age_class", 11, 4)

### 7.4 Are the per-pixel uncertainties calibrated?
Fraction of field values inside map mean ± 1.96 × map sd (ideal ≈ 95 %). The map sd is model (ensemble) uncertainty only. It excludes plot-to-pixel footprint mismatch, geolocation, and allometric error, so under-coverage is expected.

In [ ]:
coverage <- matched %>% filter(!is.na(pred_sd), in_boreal_domain) %>% tier_long() %>%
    group_by(variable, tier, source) %>%
    summarise(n = n(), coverage_95 = 100 * mean(abs(pred - ref) <= 1.96 * pred_sd),
              median_pred_sd = median(pred_sd), rmsd = sqrt(mean((pred - ref)^2)))
write_csv(coverage, file.path(OUT_DIR, "T9_uncertainty_coverage.csv"))
coverage %>% mutate(across(where(is.numeric), ~ round(.x, 1))) %>% print(n = Inf)

## 8. Sensitivity of results to screening choices
Each row changes one choice from the defaults. A conclusion is robust if bias and RMSD move less than their bootstrap CIs.

In [ ]:
sens_grid <- bind_rows(
    tibble(label = "default", mature_age = MATURE_AGE, max_gap_young = MAX_GAP_YOUNG, age_rule = AGE_RULE, use_drop = USE_DROP_FLAG),
    tibble(label = paste0("MATURE_AGE=", c(40, 80, 100)), mature_age = c(40, 80, 100), max_gap_young = MAX_GAP_YOUNG, age_rule = AGE_RULE, use_drop = USE_DROP_FLAG),
    tibble(label = paste0("MAX_GAP_YOUNG=", c(3, 10)), mature_age = MATURE_AGE, max_gap_young = c(3, 10), age_rule = AGE_RULE, use_drop = USE_DROP_FLAG),
    tibble(label = "AGE_RULE=conservative", mature_age = MATURE_AGE, max_gap_young = MAX_GAP_YOUNG, age_rule = "conservative", use_drop = USE_DROP_FLAG),
    tibble(label = "no post-2020 drop flag", mature_age = MATURE_AGE, max_gap_young = MAX_GAP_YOUNG, age_rule = AGE_RULE, use_drop = FALSE))

sens <- pmap_dfr(sens_grid, function(label, mature_age, max_gap_young, age_rule, use_drop) {
    make_pairs(plots, pred_long, mature_age, max_gap_young, age_rule, use_drop) %>%
        filter(is_matched, in_boreal_domain) %>% tier_long() %>% filter(tier != "all") %>%
        summarise_val(variable, tier, ci = FALSE) %>% mutate(setting = label)
})
# Reference-height definition (height only)
alt_mode <- if (HT_REF_MODE == "top") "legacy" else "top"
plots_alt <- plots %>% mutate(ref_ht = if (alt_mode == "top") ref_ht_top else ref_ht_legacy,
                              nonforest_ht = !is.na(ref_ht) & ref_ht < NONFOREST_HT)
sens <- bind_rows(sens,
    make_pairs(plots_alt, pred_long) %>% filter(is_matched, in_boreal_domain, variable == "ht") %>% tier_long() %>%
        summarise_val(variable, tier, ci = FALSE) %>% mutate(setting = paste0("HT_REF_MODE=", alt_mode)),
    make_pairs(plots_alt, pred_long) %>% filter(is_matched, in_boreal_domain, variable == "ht") %>%
        summarise_val(variable, source, ci = FALSE) %>% mutate(tier = "all", setting = paste0("HT_REF_MODE=", alt_mode, " by source: ", source)) %>% select(-source))
write_csv(sens, file.path(OUT_DIR, "T11_sensitivity.csv"))
sens %>% select(setting, variable, tier, n, n_plots, bias, rmsd, rrmsd, r2) %>% mutate(across(where(is.numeric), ~ round(.x, 2))) %>% print(n = Inf)

## 9. Export
Per-plot screening table (reusable for maps, e.g. the Review notebook layer) and the matched pairs.

In [ ]:
plots_out <- plots %>% select(uid, plot_id, source, group_name, region, year_obs, ref_ht, ref_ht_def, ref_agbd, ref_agbd_def,
                              field_age, age_map, age_map_sd, age_at_obs, yrs_since_obs, age_class, maturity, drop_year, excl_ht, excl_agbd) %>%
    mutate(age_class = as.character(age_class))
st_write(plots_out, file.path(OUT_DIR, "validation_plots_v3.1_screened.gpkg"), delete_dsn = TRUE, quiet = TRUE)
write_csv(pairs %>% filter(is_matched), file.path(OUT_DIR, "validation_pairs_v3.1_matched_year.csv"))
write_csv(tibble(setting = c("DATA_STAMP", "HT_REF_MODE", "MATURE_AGE", "MAX_GAP_YOUNG", "AGE_RULE", "USE_DROP_FLAG", "N_BOOT", "run_date"),
                 value = c(DATA_STAMP, HT_REF_MODE, MATURE_AGE, MAX_GAP_YOUNG, AGE_RULE, USE_DROP_FLAG, N_BOOT, format(Sys.time()))),
          file.path(OUT_DIR, "T0_run_config.csv"))
message("Outputs in ", OUT_DIR)
sessionInfo()